In [0]:
# =========================================================
# 03_GOLD_ORDERS
# =========================================================


# =========================================================
# 1. GET ACTIVE RUN
# =========================================================

running_runs = spark.sql("""
    SELECT
        run_id,
        batch_id
    FROM workspace.control.etl_run_log
    WHERE pipeline_name = 'orders_pipeline'
      AND status = 'RUNNING'
""").collect()


if len(running_runs) != 1:
    raise ValueError(
        f"Expected exactly 1 RUNNING run, "
        f"found {len(running_runs)}"
    )


run_id = running_runs[0]["run_id"]
batch_id = running_runs[0]["batch_id"]


print(f"Run ID:   {run_id}")
print(f"Batch ID: {batch_id}")


try:

    # =====================================================
    # 2. STAGE DEPENDENCY GUARDRAIL
    #
    # Gold may run ONLY after Silver completed.
    #
    # silver_rows = NULL  -> Silver was not completed
    # silver_rows = 0     -> Silver completed, but inserted 0
    # =====================================================

    run_state = spark.sql(f"""
        SELECT
            landing_rows,
            bronze_rows,
            silver_rows,
            rejected_rows,
            status
        FROM workspace.control.etl_run_log
        WHERE run_id = '{run_id}'
    """).first()


    if run_state is None:
        raise ValueError(
            f"Audit row not found for run_id={run_id}."
        )


    if run_state["silver_rows"] is None:
        raise ValueError(
            f"GOLD BLOCKED: Silver has not completed "
            f"for run_id={run_id}, "
            f"batch_id={batch_id}."
        )


    print("----------------------------------")
    print("STAGE DEPENDENCY CHECK PASSED")
    print("----------------------------------")
    print(f"Landing rows:  {run_state['landing_rows']}")
    print(f"Bronze rows:   {run_state['bronze_rows']}")
    print(f"Silver rows:   {run_state['silver_rows']}")
    print(f"Rejected rows: {run_state['rejected_rows']}")
    print("----------------------------------")


    # =====================================================
    # 3. GOLD MERGE
    # =====================================================

    spark.sql(f"""
        MERGE INTO workspace.gold.orders AS g

        USING (

            SELECT
                order_id,
                customer_id,
                amount,
                status,
                order_date,
                last_updated,
                event_id,
                batch_id

            FROM workspace.silver.orders

            WHERE batch_id = '{batch_id}'
              AND dq_overall = 1

            QUALIFY ROW_NUMBER() OVER (
                PARTITION BY order_id
                ORDER BY
                    last_updated DESC,
                    event_id DESC
            ) = 1

        ) AS s

        ON g.order_id = s.order_id


        WHEN MATCHED
             AND (
                  s.last_updated > g.last_updated

                  OR (
                      s.last_updated = g.last_updated
                      AND s.event_id > g.event_id
                  )
             )

        THEN UPDATE SET

            g.customer_id  = s.customer_id,
            g.amount       = s.amount,
            g.status       = s.status,
            g.order_date   = s.order_date,
            g.last_updated = s.last_updated,
            g.event_id     = s.event_id,
            g.batch_id     = s.batch_id,
            g.updated_at   = CURRENT_TIMESTAMP()


        WHEN NOT MATCHED

        THEN INSERT (

            order_id,
            customer_id,
            amount,
            status,
            order_date,
            last_updated,
            event_id,
            batch_id,
            updated_at

        )

        VALUES (

            s.order_id,
            s.customer_id,
            s.amount,
            s.status,
            s.order_date,
            s.last_updated,
            s.event_id,
            s.batch_id,
            CURRENT_TIMESTAMP()

        )
    """)


    print("Gold MERGE completed.")


    # =====================================================
    # 4. READ DELTA MERGE METRICS
    # =====================================================

    history = spark.sql("""
        DESCRIBE HISTORY workspace.gold.orders
    """)


    latest_merge = (
        history
        .filter(
            history.operation == "MERGE"
        )
        .orderBy(
            history.version.desc()
        )
        .first()
    )


    if latest_merge is None:
        raise ValueError(
            "No MERGE operation found in Gold history."
        )


    metrics = latest_merge["operationMetrics"]


    gold_inserted = int(
        metrics.get(
            "numTargetRowsInserted",
            0
        )
    )

    gold_updated = int(
        metrics.get(
            "numTargetRowsUpdated",
            0
        )
    )


    print(f"Gold inserted: {gold_inserted}")
    print(f"Gold updated:  {gold_updated}")


    # =====================================================
    # 5. CLOSE AUDIT AS SUCCESS
    # =====================================================

    spark.sql(f"""
        UPDATE workspace.control.etl_run_log

        SET
            gold_inserted = {gold_inserted},
            gold_updated = {gold_updated},
            end_timestamp = CURRENT_TIMESTAMP(),
            status = 'SUCCESS',
            error_message = NULL

        WHERE run_id = '{run_id}'
          AND status = 'RUNNING'
    """)


    print("Audit updated to SUCCESS.")


    # =====================================================
    # 6. FINAL AUDIT OUTPUT
    # =====================================================

    print("----------------------------------")
    print("GOLD COMPLETE")
    print("----------------------------------")
    print(f"Run ID:        {run_id}")
    print(f"Batch ID:      {batch_id}")
    print(f"Gold inserted: {gold_inserted}")
    print(f"Gold updated:  {gold_updated}")
    print("Status:         SUCCESS")
    print("----------------------------------")


    display(
        spark.sql(f"""
            SELECT
                run_id,
                batch_id,
                landing_rows,
                bronze_rows,
                silver_rows,
                rejected_rows,
                gold_inserted,
                gold_updated,
                status,
                start_timestamp,
                end_timestamp,
                error_message
            FROM workspace.control.etl_run_log
            WHERE run_id = '{run_id}'
        """)
    )


except Exception as e:

    # =====================================================
    # 7. FAILURE HANDLING
    # =====================================================

    error_message = str(e)

    print(f"GOLD FAILED: {error_message}")


    safe_error = (
        error_message
        .replace("'", "''")[:2000]
    )


    try:

        spark.sql(f"""
            UPDATE workspace.control.etl_run_log

            SET
                end_timestamp = CURRENT_TIMESTAMP(),
                status = 'FAILED',
                error_message = '{safe_error}'

            WHERE run_id = '{run_id}'
              AND status = 'RUNNING'
        """)

        print("Audit updated to FAILED.")


    except Exception as audit_error:

        print(
            f"WARNING: Failed to update audit row: "
            f"{audit_error}"
        )


    raise

Run ID:   596f9a73-298f-478d-94d2-c240a58bf207
Batch ID: batch_006
----------------------------------
STAGE DEPENDENCY CHECK PASSED
----------------------------------
Landing rows:  407
Bronze rows:   407
Silver rows:   406
Rejected rows: 2
----------------------------------
Gold MERGE completed.
Gold inserted: 322
Gold updated:  81
Audit updated to SUCCESS.
----------------------------------
GOLD COMPLETE
----------------------------------
Run ID:        596f9a73-298f-478d-94d2-c240a58bf207
Batch ID:      batch_006
Gold inserted: 322
Gold updated:  81
Status:         SUCCESS
----------------------------------


run_id,batch_id,landing_rows,bronze_rows,silver_rows,rejected_rows,gold_inserted,gold_updated,status,start_timestamp,end_timestamp,error_message
596f9a73-298f-478d-94d2-c240a58bf207,batch_006,407,407,406,2,322,81,SUCCESS,2026-09-02T18:38:12.184Z,2026-09-02T18:51:34.780Z,null


In [0]:
%sql
SELECT
    run_id,
    batch_id,
    landing_rows,
    late_arriving_rows_ingested,
    bronze_rows,
    silver_rows,
    duplicate_event_rows_removed,
    rejected_rows,
    multi_version_order_count,
    gold_inserted,
    gold_updated,
    status,
    error_message
FROM workspace.control.etl_run_log
ORDER BY start_timestamp DESC;

run_id,batch_id,landing_rows,late_arriving_rows_ingested,bronze_rows,silver_rows,duplicate_event_rows_removed,rejected_rows,multi_version_order_count,gold_inserted,gold_updated,status,error_message
596f9a73-298f-478d-94d2-c240a58bf207,batch_006,407,1,407,406,1,2,1,322,81,SUCCESS,null
6afdeae7-0039-466f-a76d-86d22d687f7c,batch_005,1,null,1,1,null,0,null,1,0,SUCCESS,null
2ac2a510-01e6-4d97-ae9a-4874bf134d7a,batch_005,2,null,2,2,null,0,null,2,0,SUCCESS,null
addbd565-7655-40eb-9477-1a924003b128,batch_005,398,null,398,398,null,0,null,298,100,SUCCESS,null
6d671f49-af7c-4b4a-971d-f95934652028,batch_004,400,null,400,400,null,0,null,300,100,SUCCESS,null
e9f04c39-297c-465c-bf2b-d6c261a6c086,batch_003,1,null,766,765,null,7,null,1,0,SUCCESS,null
c803a791-ac92-4466-8a22-297efcb338d4,batch_003,765,null,765,764,null,7,null,506,250,SUCCESS,null
8e423226-2c1e-481a-a55b-6b03339e138c,batch_003,765,null,765,764,null,6,null,null,null,FAILED,Run manually rejected for recovery. Retry requested from SILVER.
0bd99448-c8ce-40d2-8812-e7ba4aa694b2,batch_003,765,null,765,764,null,7,null,null,null,FAILED,Post-Silver validation failed: amount parser does not support currency prefix format (e.g. EUR 450). Batch requires Silver reprocessing.
cf648f21-44ea-4622-a6a9-26c9d7feed71,batch_002,null,null,null,null,null,null,null,null,null,FAILED,"[TABLE_OR_VIEW_NOT_FOUND] The table or view `workspace`.`bronze`.`orders_broken` cannot be found. Verify the spelling and correctness of the schema and catalog. Search path: [`system`.`session`, `system`.`builtin`, `system`.`ai`, `workspace`.`default`]. If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog. To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01; line 2 pos 16; InsertIntoStatement UnresolvedRelation [workspace, bronze, orders_broken], [__required_write_privileges__=INSERT], false, false, false, false, false +- Project [order_id#27336, customer_id#27337, amount#27338, status#27339, order_date#27340, last_updated#27341, event_id#27342, batch_id#27343, source_file#27344, load_timestamp#27345] +- Filter ((batch_id#27343 = batch_002) AND NOT exists#27315 [batch_id#27343 && event_id#27342]) : +- Project [1 AS 1#27356] : +- Filter ((batch_id#27353 = outer(batch_id#27343)) AND (event_id#27352 = outer(event_id#27342))) : +- SubqueryAlias b : +- SubqueryAlias workspace.bronze.orders : +- Relation workspace.bronze.orders[order_id#27346,customer_id#27347,amount#27348,status#27349,order_date#27350,last_updated#27351,event_id#27352,batch_id#27353,source_file#27354,load_timestamp#27355] parquet +- SubqueryAlias l +- SubqueryAlias workspace.landing.orders +- Relation workspace.landing.orders[order_id#27336,customer_id#27337,amount#27338,status#27339,order_date#27340,last_updated#27341,event_id#27342,batch_id#27343,source_file#27344,load_timestamp#27345] parquet JVM stacktrace: org.apache.spark.sql.catalyst.ExtendedAnalysisException at org.apache.spark.sql.errors.QueryCompilationErrors$.tableOrViewNotFoundWithSearchPath(QueryCompilationErrors.scala:1612) at org.apache.spark.sql.catalyst.analysis.package$AnalysisErrorAt.tableNotFound(package.scala:97) at org.apache.spark.sql.catalyst.analysis.CheckAnalysis.$anonfun$checkAnalysis0$1(CheckAnalysis.scala:411) at org.apache.spark.sql.catalyst.analysis.CheckAnalysis.$anonfun$checkAnalysis0$1$adapted(CheckAnalysis.scala:407) at org.apache.spark.sql.catalyst.trees.TreeNode.foreach(TreeNode.scala:372) at org.apache.spark.sql.catalyst.analysis.CheckAnalysis.checkAnalysis0(CheckAnalysis.scala:407) at org.apache.spark.sql.catalyst.analysis.CheckAnalysis.checkAnalysis0$(CheckAnalysis.scala:403) at org.apache.spark.sql.catalyst.analysis.Analyzer.checkAnalysis0(Analyzer.scala:717) at org.apache.spark.sql.catalyst.analysis.CheckAnalysis.$anonfun$checkAnalysis$1(CheckAnalysis.scala:388) at scala.runtime.java8.JFunction0$mcV$sp.apply(JFunction0$mcV$sp.scala:18) at com.d